# Attention Graph Explorer

This notebook helps debug how the bottom-up attention graph is assembled.

It is designed to answer questions like:
- Which symbols were linked into the same event cluster?
- Why did the graph connect two names?
- Which tags, peer groups, and claims created an edge?
- Which single-name claims may be hijacking a broader cluster story?

The notebook is materialized-first. It loads the latest graph-related pipeline artifacts when available.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

try:
    import networkx as nx
except Exception:
    nx = None

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name != 'streamlit_alpaca_app':
    candidate = ROOT / 'Users' / 'omai.r' / 'spectral_nature' / 'streamlit_alpaca_app'
    if candidate.exists():
        ROOT = candidate

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from services.pipeline_store import load_latest_dataset_frame

ROOT

PosixPath('/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/streamlit_alpaca_app')

## Load graph artifacts

These datasets are produced by the bottom-up attention pipeline:
- `attention_candidates_1d`
- `attention_candidate_graph`
- `attention_event_clusters_1d`
- `attention_claims`
- `attention_search_results`
- `attention_source_documents`

In [3]:
DATASET_NAMES = [
    'attention_candidates_1d',
    'attention_candidate_graph',
    'attention_event_clusters_1d',
    'attention_claims',
    'attention_search_results',
    'attention_source_documents',
]

frames = {}
metadata = {}
for dataset_name in DATASET_NAMES:
    frame, meta = load_latest_dataset_frame(dataset_name)
    frames[dataset_name] = frame
    metadata[dataset_name] = meta

pd.DataFrame(
    [
        {
            'dataset_name': name,
            'rows': int(len(frames[name])),
            'columns': list(frames[name].columns),
            'asof_time_utc': getattr(metadata[name], 'asof_time_utc', ''),
            'dataset_version_id': getattr(metadata[name], 'dataset_version_id', ''),
        }
        for name in DATASET_NAMES
    ]
)

,dataset_name,rows,columns,asof_time_utc,dataset_version_id
0,attention_candidates_1d,0,[],,
1,attention_candidate_graph,0,[],,
2,attention_event_clusters_1d,0,[],,
3,attention_claims,0,[],,
4,attention_search_results,0,[],,
5,attention_source_documents,0,[],,


In [4]:
if all(frame.empty for frame in frames.values()):
    raise RuntimeError(
        'All graph artifacts are empty. Run the attention materialization jobs first, '
        'or open this notebook from an environment that can read pipeline artifacts.'
    )

RuntimeError: All graph artifacts are empty. Run the attention materialization jobs first, or open this notebook from an environment that can read pipeline artifacts.

## Normalize JSON/list columns

In [ ]:
def parse_jsonish(value, default=None):
    if value is None:
        return [] if default is None else default
    if isinstance(value, (list, dict)):
        return value
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return [] if default is None else default
    try:
        return json.loads(text)
    except Exception:
        return [] if default is None else default


candidates = frames['attention_candidates_1d'].copy()
graph_edges = frames['attention_candidate_graph'].copy()
clusters = frames['attention_event_clusters_1d'].copy()
claims = frames['attention_claims'].copy()
search_results = frames['attention_search_results'].copy()
source_documents = frames['attention_source_documents'].copy()

for col in ['macro_exposure_tags', 'business_tags']:
    if col in candidates.columns:
        candidates[col] = candidates[col].map(parse_jsonish)

if 'edge_reasons_json' in graph_edges.columns:
    graph_edges['edge_reasons'] = graph_edges['edge_reasons_json'].map(parse_jsonish)

cluster_json_cols = [
    'member_candidate_ids_json',
    'anchor_candidate_ids_json',
    'driver_symbols_json',
    'beneficiary_symbols_json',
    'loser_symbols_json',
    'supporting_claim_ids_json',
    'event_facts_json',
]
for col in cluster_json_cols:
    if col in clusters.columns:
        clusters[col.replace('_json', '')] = clusters[col].map(parse_jsonish)

if 'claim_entities' in claims.columns:
    claims['claim_entities'] = claims['claim_entities'].map(parse_jsonish)
if 'evidence_chunk_ids' in claims.columns:
    claims['evidence_chunk_ids'] = claims['evidence_chunk_ids'].map(parse_jsonish)

print('candidates', candidates.shape)
print('graph_edges', graph_edges.shape)
print('clusters', clusters.shape)
print('claims', claims.shape)

## Candidate table

In [ ]:
candidate_view_cols = [
    'candidate_id',
    'symbol',
    'change_pct',
    'candidate_score',
    'sector',
    'industry',
    'peer_group_id',
    'macro_exposure_tags',
    'business_tags',
]
candidate_view_cols = [col for col in candidate_view_cols if col in candidates.columns]
candidates[candidate_view_cols].sort_values(['candidate_score', 'change_pct'], ascending=[False, False]).head(30)

## Edge table

Each row is a graph edge between two candidate nodes. The `edge_reasons` column is the important debug signal.

In [ ]:
candidate_lookup_cols = [
    'candidate_id',
    'symbol',
    'change_pct',
    'candidate_score',
    'sector',
    'industry',
    'peer_group_id',
    'macro_exposure_tags',
    'business_tags',
]
candidate_lookup_cols = [col for col in candidate_lookup_cols if col in candidates.columns]
candidate_lookup = candidates[candidate_lookup_cols].copy()

left_lookup = candidate_lookup.add_prefix('left_')
right_lookup = candidate_lookup.add_prefix('right_')

edges_enriched = graph_edges.merge(
    left_lookup,
    left_on='left_candidate_id',
    right_on='left_candidate_id',
    how='left',
).merge(
    right_lookup,
    left_on='right_candidate_id',
    right_on='right_candidate_id',
    how='left',
)

edge_view_cols = [
    'left_symbol',
    'right_symbol',
    'edge_weight',
    'edge_reasons',
    'left_change_pct',
    'right_change_pct',
    'left_sector',
    'right_sector',
    'left_macro_exposure_tags',
    'right_macro_exposure_tags',
]
edge_view_cols = [col for col in edge_view_cols if col in edges_enriched.columns]
edges_enriched[edge_view_cols].sort_values('edge_weight', ascending=False).head(50)

## Cluster summary

In [ ]:
def cluster_member_symbols(event_facts):
    facts = event_facts if isinstance(event_facts, dict) else {}
    members = facts.get('members') or []
    return [str(item.get('symbol', '')).upper() for item in members if str(item.get('symbol', '')).strip()]


cluster_summary = clusters.copy()
if 'event_facts' in cluster_summary.columns:
    cluster_summary['member_symbols'] = cluster_summary['event_facts'].map(cluster_member_symbols)
else:
    cluster_summary['member_symbols'] = [[] for _ in range(len(cluster_summary))]

cluster_summary['member_count'] = cluster_summary['member_symbols'].map(len)
cluster_summary['member_symbols_text'] = cluster_summary['member_symbols'].map(lambda items: ', '.join(items[:12]))

cluster_view_cols = [
    'event_id',
    'event_type',
    'cause_status',
    'event_score',
    'member_count',
    'member_symbols_text',
    'driver_symbols',
    'beneficiary_symbols',
    'loser_symbols',
]
cluster_view_cols = [col for col in cluster_view_cols if col in cluster_summary.columns]
cluster_summary[cluster_view_cols].sort_values('event_score', ascending=False).head(20)

## Symbol inspector

Set a symbol below to inspect its node, direct graph neighbors, cluster membership, and related claims.

In [ ]:
SYMBOL = 'AAL'

In [ ]:
symbol = SYMBOL.upper().strip()

symbol_row = candidates[candidates['symbol'].astype(str).str.upper() == symbol].copy()
neighbor_edges = edges_enriched[
    (edges_enriched['left_symbol'].astype(str).str.upper() == symbol)
    | (edges_enriched['right_symbol'].astype(str).str.upper() == symbol)
].copy()

def extract_other_symbol(row, target):
    left = str(row.get('left_symbol', '')).upper()
    right = str(row.get('right_symbol', '')).upper()
    return right if left == target else left

if not neighbor_edges.empty:
    neighbor_edges['other_symbol'] = neighbor_edges.apply(lambda row: extract_other_symbol(row, symbol), axis=1)
    neighbor_edges = neighbor_edges.sort_values('edge_weight', ascending=False)

cluster_hits = cluster_summary[cluster_summary['member_symbols'].map(lambda items: symbol in set(items))].copy()
claim_hits = claims[claims.get('bundle_subject', pd.Series(dtype=str)).astype(str).str.upper() == symbol].copy()

display(symbol_row[candidate_view_cols])
display(neighbor_edges[[col for col in ['other_symbol', 'edge_weight', 'edge_reasons', 'left_change_pct', 'right_change_pct', 'left_macro_exposure_tags', 'right_macro_exposure_tags'] if col in neighbor_edges.columns]].head(20))
display(cluster_hits[cluster_view_cols])
display(claim_hits[[col for col in ['claim_text', 'claim_type', 'supports_hypothesis', 'causal_score', 'confidence_score', 'is_same_day', 'claim_entities'] if col in claim_hits.columns]].sort_values(['confidence_score', 'causal_score'], ascending=False).head(20))

## Plot a local component

This uses the graph edge list. If `networkx` is installed, you get a simple spring-layout visualization.

Node color = `change_pct`

Node size = `candidate_score`

Edge hover = edge weight + edge reasons

In [ ]:
def connected_component_symbols(target_symbol: str) -> list[str]:
    target = str(target_symbol or '').upper().strip()
    if not target or graph_edges.empty:
        return []
    adjacency = {}
    for _, row in graph_edges.iterrows():
        left = str(row.get('left_symbol', '')).upper().strip()
        right = str(row.get('right_symbol', '')).upper().strip()
        if not left or not right:
            continue
        adjacency.setdefault(left, set()).add(right)
        adjacency.setdefault(right, set()).add(left)
    if target not in adjacency:
        return [target]
    stack = [target]
    seen = set()
    while stack:
        current = stack.pop()
        if current in seen:
            continue
        seen.add(current)
        stack.extend(list(adjacency.get(current, set()) - seen))
    return sorted(seen)


def plot_component(target_symbol: str):
    if nx is None:
        raise RuntimeError('networkx is not installed in this environment')

    symbols = connected_component_symbols(target_symbol)
    if not symbols:
        raise RuntimeError(f'No graph component found for {target_symbol}')

    nodes = candidates[candidates['symbol'].astype(str).str.upper().isin(symbols)].copy()
    edges = graph_edges[
        graph_edges['left_symbol'].astype(str).str.upper().isin(symbols)
        & graph_edges['right_symbol'].astype(str).str.upper().isin(symbols)
    ].copy()

    g = nx.Graph()
    for _, row in nodes.iterrows():
        symbol = str(row.get('symbol', '')).upper().strip()
        if not symbol:
            continue
        g.add_node(
            symbol,
            change_pct=float(row.get('change_pct') or 0.0),
            candidate_score=float(row.get('candidate_score') or 0.0),
            sector=str(row.get('sector', '')),
            industry=str(row.get('industry', '')),
        )
    for _, row in edges.iterrows():
        g.add_edge(
            str(row.get('left_symbol', '')).upper().strip(),
            str(row.get('right_symbol', '')).upper().strip(),
            edge_weight=float(row.get('edge_weight') or 0.0),
            edge_reasons=', '.join(row.get('edge_reasons') or []),
        )

    positions = nx.spring_layout(g, seed=7, weight='edge_weight')

    edge_x = []
    edge_y = []
    edge_text = []
    for left, right, attrs in g.edges(data=True):
        x0, y0 = positions[left]
        x1, y1 = positions[right]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_text.append(f"{left} ↔ {right}<br>weight={attrs.get('edge_weight', 0):.3f}<br>reasons={attrs.get('edge_reasons', '')}")

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        line=dict(width=1.0, color='#94a3b8'),
        hoverinfo='none',
        mode='lines',
    )

    node_x = []
    node_y = []
    node_text = []
    node_color = []
    node_size = []
    node_label = []
    for node, attrs in g.nodes(data=True):
        x, y = positions[node]
        node_x.append(x)
        node_y.append(y)
        node_label.append(node)
        node_color.append(float(attrs.get('change_pct') or 0.0))
        node_size.append(max(18.0, 18.0 + float(attrs.get('candidate_score') or 0.0) * 0.25))
        node_text.append(
            '<br>'.join([
                f"<b>{node}</b>",
                f"change_pct={float(attrs.get('change_pct') or 0.0):+.2f}%",
                f"candidate_score={float(attrs.get('candidate_score') or 0.0):.1f}",
                str(attrs.get('sector') or ''),
                str(attrs.get('industry') or ''),
            ])
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode='markers+text',
        text=node_label,
        textposition='top center',
        hovertext=node_text,
        hoverinfo='text',
        marker=dict(
            size=node_size,
            color=node_color,
            colorscale='RdYlGn',
            colorbar=dict(title='change_pct'),
            line=dict(width=1, color='#0f172a'),
        ),
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=f'Attention graph component for {target_symbol.upper()}',
        template='plotly_white',
        showlegend=False,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        margin=dict(l=20, r=20, t=60, b=20),
    )
    return fig


plot_component(SYMBOL) if nx is not None else 'Install networkx to render the graph plot.'

## Debug checklist for bad narratives

Use this notebook to inspect a weak explanation in this order:

1. Check whether the right symbols were clustered together.
2. Look at `edge_reasons` to see whether the cluster is driven by sector, tags, claim overlap, or macro bridges.
3. Inspect candidate tags like `macro_exposure_tags` and `business_tags` for missing or unsigned factor relationships.
4. Inspect symbol-level claims to see whether a single-name article is hijacking the cluster story.
5. Compare cluster members and move directions to see whether the causal sign is wrong.

Example: if oil is up and airlines are down, the graph may connect them, but it still needs signed factor logic to reason that higher fuel costs hurt airlines.